<a href="https://colab.research.google.com/github/comet-toolkit/comet_training/blob/main/CoMet_temporal_combined.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Training Session - CoMet Toolkit: Uncertainties made easy**

#Exercise 5: Dealing with uncertainties for time series

## Objectives

In this exercise you will apply the CoMet Toolkit to handle uncertainties when processing measurements separated in time one at a time:

* Combine uncertainties based on temporal error correlation
* Store uncertainty components for each processed file
* Combine the processed files at a later stage

### **Introduction**
Practically, when collecting and distributing measurement data, it is often required to process these data as they come in. Typically one or more (e.g. one day of measurements) measurements are processed from raw data, to the data product with the measurand of interest. These individual data products are then often combined at a later stage. 

In order to add uncertainties for such a use case, it is important that the error correlations are properly handled when multiple data products are combined. In order to do this, one needs to know the error correlation between the different data products (i.e. temporal error correlation). As we have seen in the previous exercise, such information can be captured in the metadata of the datafiles. It is however not always trivial to store such information, as it does not make sense to store an actual temporal error correlation matrix in each file, as we do not know a priori how many measurements there would be.

We will again use the example of the spectrometer measurements of the cress, but this time process the repeat measurements individually (imagining these would e.g. be measurements from subsequent days).

## *Step 1* - Set up the environment

First, we again install punpy.

In [ ]:
!pip install punpy>=1.0.6


Then, we clone the training repository to access the example data.

In [ ]:
!git clone https://github.com/comet-toolkit/comet_training.git

Please hit `Runtime > Restart Session` to properly load these packages into your Google Colab environment...

We also import all the relevant python packages used in this training:

In [ ]:
import xarray as xr
import numpy as np
from punpy import MCPropagation
import matplotlib.pyplot as plt

## *Step 2* - Set up input quantities and measurement function

In this step, we load the spectrometer data for the Spectralon reference panel and the cress target, copying code from the spectrometer example earlier. We also define the measurement function for converting digital numbers to reflectance.

In [ ]:
# Load datasets from NetCDF files
spectralon_ds = xr.load_dataset("comet_training/example_data/Spectralon.nc")
cress_ds = xr.load_dataset("comet_training/example_data/Cress.nc")

# Load Spectralon reference reflectance
spectralon_ref_ds = xr.load_dataset("comet_training/example_data/spectralon_reflectance.nc")

# Extract wavelengths and digital number (DN) data
wavelength = spectralon_ds['wavelength'].values

# Normalise DN measurements by integration time to make them comparable across different measurement settings
# This converts raw DN counts to a per-unit-time basis
dn_panel_repeats = spectralon_ds['digital_number'].values / spectralon_ds.attrs['integration_time_ms']      # shape: (n_wavelength, n_panel_repeats)
dn_cress_repeats = cress_ds['digital_number'].values / cress_ds.attrs['integration_time_ms']             # shape: (n_wavelength, n_cress_repeats)

def calculate_stats(dn_repeats):
    """
    Calculate mean and standard deviation across repeated measurements.

    :param dn_repeats: repeated measurements array (shape: n_wavelengths × n_repeats)
    :returns: tuple of (mean, standard_deviation) arrays
    """
    mean = np.mean(dn_repeats, axis=1)
    std = np.std(dn_repeats, axis=1, ddof=1)
    return mean, std

dn_panel_mean, dn_panel_std = calculate_stats(dn_panel_repeats)
dn_cress_mean, dn_cress_std = calculate_stats(dn_cress_repeats)

In [ ]:
# Load Spectralon reflectance
rho_panel = spectralon_ref_ds['reflectance'].values

# Create wavelength-dependent uncertainty for Spectralon reflectance (in %)
# Based on calibration certificate data
u_rho_panel_percent = np.zeros_like(wavelength)
u_rho_panel_percent[(wavelength >= 465) & (wavelength <= 780)] = 0.55
u_rho_panel_percent[(wavelength > 780) & (wavelength <= 1000)] = 1.60

# Convert uncertainty from percentage to absolute value
u_rho_panel = (u_rho_panel_percent / 100.0) * rho_panel

In [ ]:
def reflectance_model(dn_surface, dn_panel, rho_panel):
    """Calibrate surface reflectance from DN using reference panel."""
    return (dn_surface / dn_panel) * rho_panel

## *Step 3* - Process individual measurements and store results

To simulate processing measurements as they come in, we will process each repeat measurement individually, propagate uncertainties, and save the results to separate files.

In [ ]:
import obsarray

def save_results(i,rho_cress,u_random_rho_cress,u_syst_rho_cress):
    """Save results to a NetCDF file."""
    #make xarray ds
    ds = xr.Dataset(
        {
            'rho_cress': (('wavelength'), rho_cress, {"units": "1"}), #dimensionless units for reflectance
        },
        coords={'wavelength': wavelength}
    )

    #use obsarray to add random uncertainty information to the dataset (random error correlation assumed by default)
    ds.unc['rho_cress']['u_random_rho_cress'] = (('wavelength'), u_random_rho_cress, {"units": "1"}) #dimensionless units for reflectance,
    
    #define systematic error correlation structure and add systematic uncertainty information to the dataset
    err_corr_def = {
        "dim": ["wavelength"],
        "form": "systematic",
        "params": [],
        "units": []
    }
    ds.unc['rho_cress']['u_syst_rho_cress'] = (('wavelength'), u_syst_rho_cress, {"err_corr": err_corr_def,"units": "1"}) #dimensionless units for reflectance
    
    # Save to NetCDF file
    ds.to_netcdf(f'calibrated_reflectance_{i}.nc')

def load_results(i):
    """Load results from a NetCDF file."""
    ds = xr.load_dataset(f'calibrated_reflectance_{i}.nc')
    return ds['rho_cress'].values, ds['u_random_rho_cress'].values, ds['u_syst_rho_cress'].values

When we now want to propagate the uncertainties and store them in these individual files, it is useful to group them per temporal error correlation type. In our case, we have temporally random error correlations and temporally systematic. We can propagate these separately, adding in all contributions for that group. In our case there are few, but this could require already combining multiple uncertainties on the inputs, and potentially dealing with error correlation along any other dimensions that the data might have. 

In [ ]:
# Initialise punpy's Monte Carlo propagation object with 10,000 iterations
prop = MCPropagation(100)

for i in range(len(dn_cress_repeats[0])):
    rho_cress = reflectance_model(dn_cress_repeats[:,i], dn_panel_mean, rho_panel)
    u_random_rho_cress = prop.propagate_random(
    reflectance_model,
        [dn_cress_repeats[:,i], dn_panel_mean, rho_panel],
        [dn_cress_std, dn_panel_std, None]
    )
    u_syst_rho_cress = prop.propagate_systematic(
        reflectance_model,
        [dn_cress_repeats[:,i], dn_panel_mean, rho_panel],
        [None, None, u_rho_panel]
    )
    save_results(i,rho_cress,u_random_rho_cress,u_syst_rho_cress)


## *Step 3* - Combine the processed measurements

Now that we have processed each individual measurement and stored the uncertainty components, we can combine them at a later stage. When combining measurements from different times, it is important to consider the temporal error correlation.

In [ ]:
rho_cress = np.empty((len(wavelength), len(dn_cress_repeats[0])))
u_random_rho_cress = np.zeros_like(rho_cress)
u_syst_rho_cress = np.zeros_like(rho_cress)

for i in range(len(dn_cress_repeats[0])):
    rho_cress[:,i] = load_results(i)[0]
    u_random_rho_cress[:,i] = load_results(i)[1]
    u_syst_rho_cress[:,i] = load_results(i)[2]

def calculate_mean(data):
    return np.mean(data, axis=1)

# Calculate the average reflectance
rho_cress_avg = calculate_mean(rho_cress)

rho_cress_avg_ur = prop.propagate_random(calculate_mean,[rho_cress],
      [u_random_rho_cress])
print("average reflectance random uncertainty:", rho_cress_avg_ur)
rho_cress_avg_us = prop.propagate_systematic(calculate_mean,[rho_cress],
      [u_syst_rho_cress])
print("average reflectance systematic uncertainty:", rho_cress_avg_us)
rho_cress_avg_ut_errcorr = (rho_cress_avg_ur**2 + rho_cress_avg_us**2)**0.5
print("average reflectance total uncertainty:", rho_cress_avg_ut_errcorr)

In [ ]:
    # Plot the average reflectance with uncertainty
plt.figure(figsize=(10, 6))
plt.plot(wavelength, rho_cress_avg, color='red', linewidth=2, label='Average reflectance')
plt.fill_between(wavelength, rho_cress_avg - rho_cress_avg_ut_errcorr, rho_cress_avg + rho_cress_avg_ut_errcorr, alpha=0.3, color='red', label='Total uncertainty')
plt.fill_between(wavelength, rho_cress_avg - rho_cress_avg_ur, rho_cress_avg + rho_cress_avg_ur, alpha=0.3, color='blue', label='Random uncertainty')
plt.fill_between(wavelength, rho_cress_avg - rho_cress_avg_us, rho_cress_avg + rho_cress_avg_us, alpha=0.3, color='green', label='Systematic uncertainty')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Reflectance')
plt.title('Average Cress reflectance with uncertainty components')
plt.legend()
plt.show()

## *Step 4* - Storing uncertainty components for multi-temporal analysis

When processing time series data, it is often necessary to store the uncertainty information in a way that allows for proper combination later. This is particularly important when the data will be used in higher-level products or when temporal correlations need to be accounted for.

In this exercise, we stored the random and systematic uncertainty components separately for each processed file. This allows us to:

1. **Combine uncertainties correctly**: When averaging multiple measurements, we can propagate the random uncertainties assuming they are uncorrelated, and the systematic uncertainties assuming they are fully correlated.

2. **Maintain traceability**: By keeping uncertainty components separate, we can trace back which sources contribute most to the final uncertainty.

3. **Enable flexible analysis**: Different analysis scenarios may require different assumptions about error correlations.

The approach demonstrated here can be extended to more complex scenarios, such as combining data from multiple sensors or processing levels. 

When there are many uncertainty components, it is often not necessary to save each of them separately. You can combine them per type of temporal error correlation (often just random and systematic). If relevant, store (combined) error correlation along other dimensions (e.g. spectral error correlation) within the product files.